**Age Model Architecture**

input: image

output: age

regression model

Dataset: UTKFace
Link to download:

part 1: https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669

part 2: https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426


part 3: https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395

### Get the Data

In [ ]:
#!rm -rf datasets/

In [ ]:
# Download and unzip the datasets from UTKface

import tarfile
import urllib.request
import os

download_url_part1 = "https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669"
download_url_part2 = "https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426"
download_url_part3 = "https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395"

# Download the datasets
download_urls = [download_url_part1, download_url_part2, download_url_part3]
face_path = os.path.join("datasets", "faces")
os.makedirs(face_path, exist_ok=True)

for dataset_num in range(len(download_urls)):
  dataset_folder_name = "part" + str(dataset_num + 1)
  if (not os.path.exists(os.path.join(face_path, dataset_folder_name)) and not os.path.exists(os.path.join(face_path, "images"))):
    print(f"Extracting dataset {dataset_num+1}...")
    faces_tar_location = os.path.join(face_path, "faces" + str(dataset_num+1) + ".tgz")
    urllib.request.urlretrieve(download_urls[dataset_num], faces_tar_location)

    # extract tar dataset
    faces_tgz = tarfile.open(faces_tar_location)
    faces_tgz.extractall(path="datasets/faces")
    faces_tgz.close()

    # remove tar file
    os.remove(faces_tar_location)

  else:
    print(f"dataset {dataset_num+1} already exists")



In [ ]:
# Get IMBD Data

download_url = "https://data.vision.ee.ethz.ch/cvl/rrothe/imdb-wiki/static/wiki_crop.tar"

imdb_raw_data_location = os.path.join(face_path, "imbd-face-dataset.tgz")
urllib.request.urlretrieve(download_url, imdb_raw_data_location)

imdb_faces_tgz = tarfile.open(imdb_raw_data_location)
imdb_faces_tgz.extractall(path="datasets/faces")
imdb_faces_tgz.close()

os.remove(imdb_raw_data_location)

In [ ]:
# Combine all data into one folder
import shutil

faces_images_path = os.path.join(face_path, "images")
os.makedirs(faces_images_path, exist_ok=True)

# Get source folders to copy image data from
source_folders = []
for dataset in range(len(download_urls)):
  dataset_path = os.path.join("./datasets/faces/part" + str(dataset+1))
  if (os.path.exists(dataset_path)):
    source_folders.append(dataset_path)

for folder in source_folders:
  file_names = os.listdir(folder)
  for file_name in file_names:
    shutil.move(os.path.join(folder, file_name), faces_images_path)
  os.rmdir(folder) # remove directory because we don't need it anymore

In [ ]:
wiki_faces_path = os.path.join(face_path, "wiki_crop_processed")
os.makedirs(wiki_faces_path, exist_ok=True)

# Get source folders to copy image data from
source_folders = []
for dataset in range(0,100):
    dataset_path = os.path.join("./datasets/faces/wiki_crop", str(dataset))
    if (os.path.exists(dataset_path)):
        source_folders.append(dataset_path)

for folder in source_folders:
    wiki_file_names = os.listdir(folder)
    for wiki_file_name in wiki_file_names:
        file_split = wiki_file_name.split("_")
        unique_num = file_split[0]
        birth_year, photo_taken_year = file_split[1].split("-")[0], file_split[2].split(".")[0]
        age_in_photo = int(photo_taken_year) - int(birth_year)
        if (age_in_photo >= 0):
            new_file_name = f"{age_in_photo}_{unique_num}.jpg"
            os.rename(os.path.join(folder, wiki_file_name), os.path.join(folder, new_file_name))
            shutil.move(os.path.join(folder, new_file_name), wiki_faces_path)



In [ ]:
!pip3 install pillow rich rich-pixels
!pip3 install setuptools==81.0
!pip3 install face-recognition
!pip3 install git+https://github.com/ageitgey/face_recognition_models

In [ ]:
file_names = os.listdir(faces_images_path)
print(f"Number of images: {len(file_names)}")

wiki_file_names = os.listdir(wiki_faces_path)
print(f"Number of images: {len(wiki_file_names)}")

### Cropping Images & Adding Padding

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import importlib
import face_processor

# Reload to pick up face_processor.py edits without restarting kernel manually.
importlib.reload(face_processor)
process_single_image = face_processor.process_single_image

os.makedirs(os.path.join(face_path, "cropped"), exist_ok=True)
test_set = [file_names, wiki_file_names] # test batch
dir_basenames = ["images", "wiki_crop_processed"]

num_workers = 25
images_already_processed = set(os.listdir("./datasets/faces/cropped"))

for i in range(1, 2):
  pending_images = [img for img in test_set[i] if img not in images_already_processed]
  print(f"Submitting {len(pending_images)} images from {dir_basenames[i]}...")

  results = []
  with ProcessPoolExecutor(max_workers=num_workers) as ppe:
      futures = [ppe.submit(process_single_image, img, dir_basenames[i]) for img in pending_images]
      for fut in as_completed(futures):
        try:
          results.append(fut.result())
        except Exception as err:
          # Do not abort the full batch when one worker/task fails.
          results.append(("<unknown>", False, f"future_error:{type(err).__name__}"))

  num_success = sum(1 for _, success_boolean, _ in results if success_boolean)
  num_no_face = sum(1 for _, success_boolean, msg in results if (not success_boolean and msg == "no_face"))
  num_errors = sum(1 for _, success_boolean, msg in results if (not success_boolean and msg.startswith("error:")))

  print(f"Finished {len(results)} files from {dir_basenames[i]}")
  print(f"Saved faces: {num_success} | No face: {num_no_face} | Errors: {num_errors}")

In [ ]:
import random

# Shuffling image data before splitting it

random.seed(49328042)
images_shuffled = os.listdir("./datasets/faces/wiki_crop_processed")
random.shuffle(images_shuffled)

age_labels = []

for file in images_shuffled:
    age_label = int(file[0:file.index('_')])
    age_labels.append(age_label)


In [ ]:
# Split the data into training, validation, and testing sets

train_data = images_shuffled[:25000]
train_labels = age_labels[:25000]

validation_data = images_shuffled[25001:28000]
validation_labels = age_labels[25001:28000]

testing_data = images_shuffled[28001:]
testing_labels = age_labels[28001:]


In [ ]:
# Create image set directories based on labels

import os, shutil, pathlib

original_dir = "./" + str(pathlib.Path("datasets/faces/wiki_crop_processed"))
base_training_dir = "./" + str(pathlib.Path("datasets/faces/training"))

def make_data_sets(subdir, starting_index, ending_index):
    for img_index in range(starting_index, ending_index):
        dir_path = f"{base_training_dir}/{subdir}/{str(age_labels[img_index])}"
        os.makedirs(dir_path, exist_ok=True)

        image_path = os.path.join(original_dir, images_shuffled[img_index])
        if (image_path not in os.listdir(dir_path)):
            shutil.move(image_path, dir_path)

if (len(os.listdir(original_dir)) > 0):
    make_data_sets("train", 0, 25000)
    make_data_sets("validation", 25001, 28000)
    make_data_sets("test", 28001, 38841)



In [ ]:
# set max thread count for model training

import os

os.environ["OMP_NUM_THREADS"] = "26"
os.environ["TF_NUM_INTRAOP_THREADS"] = "26"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"


In [ ]:
# Create datasets based on directories
from tensorflow.keras.utils import image_dataset_from_directory

batch_size = 32

# 16650 files
train_dataset = image_dataset_from_directory(
    f"{base_training_dir}/train",
    image_size=(256,256),
    batch_size=batch_size,
)

# 2400 files
validation_dataset = image_dataset_from_directory(
    f"{base_training_dir}/validation",
    image_size=(256,256),
    batch_size=batch_size
)

# 5000 files
test_dataset = image_dataset_from_directory(
    f"{base_training_dir}/test",
    image_size=(256,256),
    batch_size=batch_size,
)

In [ ]:
for data_batch, labels_batch in train_dataset:
    print(f"Shape of data set: {data_batch.shape}")
    print(f"Shape of label set: {labels_batch.shape}")
    break

### Building the CNN

In [ ]:
%pip install tensorflow
%pip install numpy

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import Sequential

# 1 Input Layer
# 3 Convolutional Layers
# 3 Max Pooling Layers
# 2 Dense Layers (after flattening)

cnn_model = Sequential([
    layers.Input(shape=(256,256,3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(256, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Conv2D(512, (3,3), activation="relu"),
    layers.MaxPool2D((2,2), 2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1)
])

cnn_model.summary()


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs avaliable: {len(gpus)}")
for gpu in gpus:
    print(f"gpu: {gpu}")

In [ ]:
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable

# Custom metric to track percent of age predictions correct within N-year tolerance.
@register_keras_serializable(package="Custom")
class WithinNYears(tf.keras.metrics.Metric):
    def __init__(self, tolerance=None, dtype=None, name=None, **kwargs):
        # Backward-compatible load path: if older saved configs omit tolerance
        if tolerance is None and isinstance(name, str):
            match = re.match(r"within_(\d+)_years", name)
            if match:
                tolerance = int(match.group(1))

        if tolerance is None:
            tolerance = 2

        super().__init__(name=name or f"within_{tolerance}_years", dtype=dtype, **kwargs)
        self.tolerance = int(tolerance)
        self.correct = self.add_weight(name="correct", initializer="zeros")
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.squeeze(y_pred, axis=-1) # get last tensor (output) and transform (batch_size,1) to (batch_size,)
        y_true = tf.cast(y_true, tf.float32) 
        within = tf.abs(y_true - y_pred) <= self.tolerance
        self.correct.assign_add(tf.reduce_sum(tf.cast(within, tf.float32))) # convert booleans to floats and get total sum and add to correct
        self.total.assign_add(tf.cast(tf.size(y_true), tf.float32)) # total is just size of y_true

    def result(self):
        return self.correct / self.total
    
    def reset(self): # runs once per epoch
        self.correct.assign(0)
        self.total.assign(0)

    def get_config(self):
        config = super().get_config()
        config.update({"tolerance": self.tolerance})
        return config

In [ ]:
# compiling model and adding callbacks

cnn_model.compile(
    loss=tf.keras.losses.Huber(delta=3.0),
    optimizer=tf.keras.optimizers.AdamW(),
    metrics=["mae", WithinNYears(2), WithinNYears(5), WithinNYears(10)]
)

# saves best model
# stops if no improvement after 6 epochs

callbacks = [
    keras.callbacks.ModelCheckpoint( 
        filepath="cnn_model_best.keras",
        save_best_only=True,
        monitor="val_mae",
        mode="min",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=6,
        restore_best_weights=True,
        verbose=1
    )
]



In [ ]:
training_history = cnn_model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks
)

### Tuning Model

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    mae = history.history["mae"]
    val_mae = history.history["val_mae"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(mae) + 1)

    plt.plot(epochs, mae, "bo", label="Training Mae")
    plt.plot(epochs, val_mae, "b", label="Validation Mae")
    plt.title("Training and Validation MAE")
    plt.xlabel("Epochs")
    plt.ylabel("MAE")
    plt.legend()
    plt.figure()

    plt.plot(epochs[1:], loss[1:], "bo", label="Training Loss")
    plt.plot(epochs[1:], val_loss[1:], "b", label="Validation Loss")
    plt.title("Training and Validation Loss")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


In [ ]:
plot_training_history(training_history)

### Evaluating the Model

In [ ]:
custom_objects = {"WithinNYears": WithinNYears}
test_model = keras.models.load_model("cnn_model_best.keras", custom_objects=custom_objects)
test_loss = test_model.evaluate(test_dataset)
print(f"Test Loss: {test_loss[0]:.3f}\nTest MAE: {test_loss[1]:.3f}\nTest within 10y: {test_loss[4]:.3f}\nTest within 5y: {test_loss[3]:.3f}\nTest within 2y: {test_loss[2]:.3f}")

In [ ]:
from PIL import Image, ImageOps
import face_recognition
import os

!pwd

image = face_recognition.load_image_file(os.path.join("./", "lisa.jpg"))
face_locations = face_recognition.face_locations(image, model="hog")

if len(face_locations) > 0: # face found in image

    # Get Cropped Image
    height, width = image.shape[:2]
    top, right, bottom, left = face_locations[0]
    
    box_h = bottom - top
    box_w = right - left
    pad_h = int(box_h * 0.33) # add padding to get hair, neck, etc.
    pad_w = int(box_w * 0.33)

    top = max(0, top-pad_h)
    bottom = min(height, bottom+pad_h)
    left = max(0, left-pad_w)
    right = min(width, right+pad_w)

    face_image = image[top:bottom, left:right]
    pil_image = Image.fromarray(face_image)

    # add padding
    processed_image = ImageOps.pad(
        pil_image,
        (256,256),
        Image.Resampling.LANCZOS,
        (0,0,0) # black padding
    )

    processed_image.save(f"lisa_cropped.jpg")

In [ ]:
import numpy as np
from PIL import Image
from tensorflow import keras
import os

img_path = "lisa_cropped.jpg"
model_path = "cnn_model_best.keras"

if not os.path.exists(img_path):
    raise FileNotFoundError(f"Could not find {img_path}. Run the crop cell first.")

if "test_model" not in globals():
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Could not find {model_path}.")
    test_model = keras.models.load_model(model_path, compile=False)

img = Image.open(img_path)
img_array = np.array(img, dtype=np.float32)
img_batch = np.expand_dims(img_array, axis=0)

print(f"Input tensor shape: {img_batch.shape}")

pred = test_model.predict(img_batch, verbose=0)
predicted_age = float(pred[0][0])
print(f"Predicted age: {predicted_age:.2f}")